In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import scipy.stats as stats

# 1. 데이터 불러오기

In [20]:
df = pd.read_csv('./data/telecom_encoding_final.csv')
df.head()

,id,gender,age,year,school,income,job,phone_usage_per_m,telecom_change_yn,mobile_bundle,...,jeonbuk,jeonnam,gyeongbuk,gyeongnam,jeju,sejong,skt,kt,lgu,mvno
0,10002,1,45,2017,4,0,0,29,0,0,...,0,0,0,0,0,0,1,0,0,0
1,10002,1,46,2018,4,0,0,35,0,1,...,0,0,0,0,0,0,1,0,0,0
2,10002,1,47,2019,4,0,0,58,0,1,...,0,0,0,0,0,0,1,0,0,0
3,10002,1,48,2020,4,0,0,37,1,1,...,0,0,0,0,0,0,0,1,0,0
4,10002,1,49,2021,4,0,0,32,0,1,...,0,0,0,0,0,0,0,1,0,0


In [25]:
df['telecom_change_yn'].value_counts()

telecom_change_yn
0    19837
1    11735
Name: count, dtype: int64

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31572 entries, 0 to 31571
Data columns (total 35 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 31572 non-null  int64
 1   gender             31572 non-null  int64
 2   age                31572 non-null  int64
 3   year               31572 non-null  int64
 4   school             31572 non-null  int64
 5   income             31572 non-null  int64
 6   job                31572 non-null  int64
 7   phone_usage_per_m  31572 non-null  int64
 8   telecom_change_yn  31572 non-null  int64
 9   mobile_bundle      31572 non-null  int64
 10  mar_1              31572 non-null  int64
 11  mar_2              31572 non-null  int64
 12  mar_3              31572 non-null  int64
 13  mar_4              31572 non-null  int64
 14  seoul              31572 non-null  int64
 15  busan              31572 non-null  int64
 16  daegu              31572 non-null  int64
 17  incheon     

# 2. XGBoost

### 2-1. 첫 번째 시도

- xgb 정확도가 0.68로 떨어짐

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

X = df.drop(['telecom_change_yn'], axis=1)
y = df['telecom_change_yn']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=24)

xgb_clf = XGBClassifier(
    n_estimators=600,
    learning_rate=0.,
    max_depth=10,
    random_state=42,
    eval_metric='logloss'
)

xgb_clf.fit(X_train, y_train)

print(xgb_clf.score(X_train, y_train), xgb_clf.score(X_test, y_test))

feat_impt_ser = pd.Series(xgb_clf.feature_importances_, index=X.columns).sort_values(ascending=False)

xgb_pred = xgb_clf.predict(X_test)
print(f"xgb 정확도: {accuracy_score(y_test, xgb_pred):.4f}")
print(classification_report(y_test, xgb_pred))

0.6303475653532666 0.6221968833143292
xgb 정확도: 0.6222
              precision    recall  f1-score   support

           0       0.62      1.00      0.77      4911
           1       0.00      0.00      0.00      2982

    accuracy                           0.62      7893
   macro avg       0.31      0.50      0.38      7893
weighted avg       0.39      0.62      0.48      7893



c:\Users\yunu2\miniconda3\envs\mlstudy_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\yunu2\miniconda3\envs\mlstudy_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\yunu2\miniconda3\envs\mlstudy_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize

### 2-2. 두 번째 시도

- 이탈하지 않는 고객은 잘 맞춤 (0-recall: 0.88)
- 이탈 고객은 많이 놓침
    - Precision: 0.68
    - Recall: 0.37
    - F1: 0.47
-> 보완이 필요함

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

X = df.drop(['telecom_change_yn'], axis=1)
y = df['telecom_change_yn']

# Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=24,
    stratify=y
)

# 클래스 불균형 보정 (있다면)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# 과적합 방지 세팅
xgb_clf = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=5,
    gamma=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2,
    random_state=42,
    eval_metric='logloss',
    early_stopping_rounds=50
)

xgb_clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# 평가
train_score = xgb_clf.score(X_train, y_train)
test_score = xgb_clf.score(X_test, y_test)

proba = xgb_clf.predict_proba(X_test)[:,1]
roc_auc = roc_auc_score(y_test, proba)

print(f"Train Accuracy: {train_score:.4f}")
print(f"Test Accuracy:  {test_score:.4f}")
print(f"ROC-AUC:        {roc_auc:.4f}")

y_pred = xgb_clf.predict(X_test)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Train Accuracy: 0.7382
Test Accuracy:  0.6868
ROC-AUC:        0.7218

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.88      0.78      3968
           1       0.64      0.37      0.47      2347

    accuracy                           0.69      6315
   macro avg       0.67      0.62      0.62      6315
weighted avg       0.68      0.69      0.66      6315



### 2-3. 세 번째 시도

- threshold tuning
- 클래스 불균형 보정
- 과적합 방지로 early stopping 활용

In [24]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, precision_recall_curve, confusion_matrix
from xgboost import XGBClassifier

X = df.drop(['telecom_change_yn'], axis=1)
y = df['telecom_change_yn']

# Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=24,
    stratify=y
)

# 2) 클래스 불균형 보정
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

# 3) 과적합 방지 + PR-AUC 최적화
xgb_clf = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=5,
    gamma=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,      # 규제 추가
    reg_lambda=2,       # 규제 추가
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='aucpr',
    early_stopping_rounds=50    #early stopping 적용 
)

xgb_clf.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# 4) 확률 기반 평가
proba = xgb_clf.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, proba)

print(f"ROC-AUC: {roc_auc:.4f}")
print("best_iteration:", getattr(xgb_clf, "best_iteration", None))

# 5) Threshold 튜닝 (recall 목표 기반)
prec, rec, thr = precision_recall_curve(y_test, proba)

target_recall = 0.60  # 너가 원했던 수준 (원하면 0.65/0.70로 바꿔)
idx = np.where(rec >= target_recall)[0]
best_idx = idx[np.argmax(prec[idx])]  # recall 조건 만족 중 precision 최대

best_threshold = thr[best_idx - 1] if best_idx > 0 else 0.5
print(f"추천 threshold: {best_threshold:.6f}")
print(f"precision: {prec[best_idx]:.4f}, recall: {rec[best_idx]:.4f}")

# 6) threshold 적용 예측
y_pred = (proba >= best_threshold).astype(int)

# 7) 최종 리포트 (이게 너가 발표/보고서에 쓰는 결과)
print("\nClassification Report (threshold 적용):")
print(classification_report(y_test, y_pred))

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

scale_pos_weight: 1.6903493821900297
ROC-AUC: 0.7242
best_iteration: 989
추천 threshold: 0.518734
precision: 0.5625, recall: 0.6003

Classification Report (threshold 적용):
              precision    recall  f1-score   support

           0       0.75      0.72      0.74      3968
           1       0.56      0.60      0.58      2347

    accuracy                           0.68      6315
   macro avg       0.66      0.66      0.66      6315
weighted avg       0.68      0.68      0.68      6315

Accuracy: 0.6777513855898654
Confusion Matrix:
 [[2871 1097]
 [ 938 1409]]


### 2-4. 네 번째 시도

In [26]:
!pip install lightgbm